In [ ]:
# Steps to follow for the project:

# 1. Import libraries 
# 2. Load dataset 
# 3. Explore the data (EDA)
# 4. Clean the data (handle missing values, convert text to numbers)  this was missing
# 5. Declare X and y
# 6. Split into train/test
# 7. Create the model
# 8. Fit the model on training data
# 9. Predict on the test set, then evaluate (predicted vs actual this covers your old steps 8 and 9 in one)
# 10. Compare with a Random Forest model

# DEMO: Predict rent for a single house you describe yourself

In [1]:
# Step 1: Import libraries needed
!pip install pandas scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Step 2: Load the dataset
df_rent = pd.read_csv('nigeria_rentals_10k.csv')
print('load successfully')


# step 3: Explore the dataset or features
# Shape and structure
print("Rows, Columns:", df_rent.shape)
df_rent.info()

# Missing values — how much is missing, per column
print("\nMissing values per column:")
print(df_rent.isnull().sum())

# Basic stats on the number columns (rent, bedrooms, bathrooms, size)
df_rent.describe()



load successfully
Rows, Columns: (10706, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10706 entries, 0 to 10705
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   location       10706 non-null  object 
 1   city           10706 non-null  object 
 2   property_type  10706 non-null  object 
 3   bedrooms       10706 non-null  int64  
 4   bathrooms      10706 non-null  int64  
 5   size_sqm       7665 non-null   float64
 6   furnished      10706 non-null  object 
 7   serviced       10706 non-null  object 
 8   annual_rent    10706 non-null  int64  
 9   listing_date   10706 non-null  object 
 10  listing_url    10706 non-null  object 
dtypes: float64(1), int64(3), object(7)
memory usage: 920.2+ KB

Missing values per column:
location            0
city                0
property_type       0
bedrooms            0
bathrooms           0
size_sqm         3041
furnished           0
serviced            0
annual

,bedrooms,bathrooms,size_sqm,annual_rent
count,10706.000000,10706.000000,7665.000000,1.070600e+04
mean,2.577527,2.750234,371.121331,5.072165e+06
std,1.601557,1.554194,305.665128,3.189454e+06
min,0.000000,1.000000,40.000000,-1.000000e+05
25%,2.000000,1.000000,104.000000,3.100000e+06
50%,3.000000,3.000000,267.000000,4.700000e+06
75%,4.000000,4.000000,619.000000,6.300000e+06
max,6.000000,7.000000,1200.000000,2.020000e+07


In [ ]:
# Check for impossible values before cleaning
print("Rows with annual_rent <= 0:", (df_rent['annual_rent'] <= 0).sum())
print(df_rent[df_rent['annual_rent'] <= 0][['location','city','property_type','bedrooms','annual_rent']].head())

Rows with annual_rent <= 0: 136
          location   city property_type  bedrooms  annual_rent
88   Lekki Phase 2  Lagos  self-contain         0            0
237       Lokogoma  Abuja  self-contain         0            0
293            Apo  Abuja  self-contain         0            0
334          Garki  Abuja  self-contain         0      -100000
425          Dutse  Abuja  self-contain         0            0


In [ ]:
# STEP 4: Clean the data

# 1. Remove impossible rent values (rent can't be zero or negative)
df_clean = df_rent[df_rent['annual_rent'] > 0].copy()
print(f"Removed {len(df_rent) - len(df_clean)} rows with invalid rent")

# 2. Fill missing size_sqm with the median size for that property_type
#    (median is safer than mean here since size has big outliers, e.g. mansions)
df_clean['size_sqm'] = df_clean.groupby('property_type')['size_sqm'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Drop columns that don't help predict price
df_clean.drop(columns=['listing_date', 'listing_url'], inplace=True)

# 4. Convert text columns into numbers so the model can read them
#    pd.get_dummies turns each category into its own 0/1 column
df_clean = pd.get_dummies(
    df_clean,
    columns=['location', 'city', 'property_type', 'furnished', 'serviced'],
    drop_first=True  # avoids redundant columns
)

print("Missing values left:", df_clean.isnull().sum().sum())
print("Shape after cleaning:", df_clean.shape)
df_clean.head()

Removed 136 rows with invalid rent
Missing values left: 0
Shape after cleaning: (10570, 49)


,bedrooms,bathrooms,size_sqm,annual_rent,location_Allen,location_Amuwo Odofin,location_Apo,location_Asokoro,location_Dutse,location_Festac,...,property_type_duplex,property_type_flat,property_type_mansion,property_type_penthouse,property_type_self-contain,property_type_terrace,furnished_not stated,furnished_yes,serviced_not stated,serviced_yes
0,2,1,115.0,3600000,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,True
1,2,1,140.0,4800000,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,4,5,280.5,4500000,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
3,1,2,100.0,1700000,False,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,True
4,3,2,1156.0,6300000,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,True


In [ ]:
# STEP 5: Declare X (features/inputs) and y (target/what we're predicting)

X = df_clean.drop(columns=['annual_rent'])  # everything except the price = inputs
y = df_clean['annual_rent']                  # the price = what we want to predict

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10570, 48)
y shape: (10570,)


In [ ]:
# STEP 6: Split into training and test sets
# from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # hold back 20% of the data for testing
    random_state=42    # fixes the random shuffle so results are reproducible every run
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (8456, 48) (8456,)
Test set: (2114, 48) (2114,)


In [ ]:
# STEP 7: Create the model
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [ ]:
# STEP 8: Fit the model — this is where the model actually "learns"

model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


In [ ]:
# STEP 9: Predict on test data, then check how close the guesses were to reality

from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)  # model's guesses for the 2,114 listings it has never seen

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Average error (MAE): ₦{mae:,.0f}")
print(f"R² score: {r2:.3f}")

# Look at a few real vs predicted side by side
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10].round(0)
})
print(comparison)

Average error (MAE): ₦941,691
R² score: 0.849
    Actual   Predicted
0  8400000  10590002.0
1  5200000   6480708.0
2  4200000   5122327.0
3  6300000   6745607.0
4  8100000   8446478.0
5  5400000   5082453.0
6  8500000   9165035.0
7  6200000   6002805.0
8  4300000   4275902.0
9  1900000   1530075.0


In [ ]:
# STEP 10: Compare with a Random Forest model

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Linear Regression -> MAE: ₦{:,.0f}  |  R²: {:.3f}".format(mae, r2))
print("Random Forest     -> MAE: ₦{:,.0f}  |  R²: {:.3f}".format(mae_rf, r2_rf))

Linear Regression -> MAE: ₦941,691  |  R²: 0.849
Random Forest      -> MAE: ₦527,603  |  R²: 0.963


In [ ]:
# DEMO CELL: Predict rent for a single house you describe yourself
def predict_rent(location, city, property_type, bedrooms, bathrooms, size_sqm, furnished, serviced):
    """
    Type in a house's details in plain terms, get back a predicted rent.
    Example values:
      location      = "Ikeja", "Lekki Phase 1", "Wuse 2", etc. (must match a value in df_rent['location'])
      city          = "Lagos" or "Abuja"
      property_type = "flat", "duplex", "bungalow", "self-contain", "mansion", "terrace", "penthouse"
      bedrooms      = a number, e.g. 3
      bathrooms     = a number, e.g. 2
      size_sqm      = a number, e.g. 150
      furnished     = "yes", "no", or "not stated"
      serviced      = "yes", "no", or "not stated"
    """
    # Start with a single row of all zeros, matching every column the model was trained on
    row = pd.DataFrame(0, index=[0], columns=X_train.columns)

    # Fill in the plain number columns
    row.loc[0, 'bedrooms'] = bedrooms
    row.loc[0, 'bathrooms'] = bathrooms
    row.loc[0, 'size_sqm'] = size_sqm


    for col_prefix, value in [
        ('location', location),
        ('city', city),
        ('property_type', property_type),
        ('furnished', furnished),
        ('serviced', serviced),
    ]:
        col_name = f"{col_prefix}_{value}"
        if col_name in row.columns:
            row.loc[0, col_name] = 1

    predicted_price = rf_model.predict(row)[0]
    print(f"Predicted annual rent: ₦{predicted_price:,.0f}")
    return predicted_price


#Example: try it live in your demo
predict_rent(
    location="Ikeja",
    city="Lagos",
    property_type="flat",
    bedrooms=3,
    bathrooms=2,
    size_sqm=150,
    furnished="yes",
    serviced="yes"
)

Predicted annual rent: ₦3,705,000


np.float64(3705000.0)